# Multi-View Recommendation System - DDP Hybrid Mode

**架构说明**: 本notebook采用混合架构，结合notebook交互性和DDP高性能训练：

- **Notebook中执行**: 数据预处理、参数配置、结果查看、可视化
- **DDP脚本执行**: GPU密集型训练（通过`torchrun`调用）

## 优势

✅ **保持交互性**: 在Jupyter中运行，可随时查看中间结果
✅ **完整DDP支持**: 关键训练步骤使用多GPU分布式训练
✅ **灵活调试**: 可以单独运行或调试某个步骤
✅ **独立脚本**: DDP脚本可独立运行，便于生产环境部署

## DDP脚本

- `ddp_scripts/step6_sgns_training_ddp.py` - SGNS嵌入训练（最耗时）
- `ddp_scripts/step7_faiss_ann_ddp.py` - FAISS k-NN图构建

## GPU配置检测


In [1]:
# 检测GPU配置
import torch
import subprocess

NUM_GPUS = torch.cuda.device_count()
print(f"可用GPU数量: {NUM_GPUS}")

if NUM_GPUS > 0:
    for i in range(NUM_GPUS):
        props = torch.cuda.get_device_properties(i)
        mem_gb = props.total_memory / (1024**3)
        print(f"  GPU {i}: {props.name}, {mem_gb:.1f} GB")

    # 建议的nproc_per_node
    print(f"\n建议的torchrun参数: --nproc_per_node={NUM_GPUS}")
else:
    print("警告: 未检测到GPU，DDP脚本将无法运行")

# 检查torchrun是否可用
try:
    result = subprocess.run(['torchrun', '--help'], capture_output=True, timeout=5)
    print("\n✓ torchrun 可用")
except Exception as e:
    print(f"\n✗ torchrun 不可用: {e}")
    print("  请安装: pip install torch --upgrade")

/opt/conda/envs/recsys-gpu/lib/python3.10/site-packages/torch/cuda/__init__.py:1007: UserWarning: Can't initialize NVML
  raw_cnt = _raw_device_count_nvml()


可用GPU数量: 0
警告: 未检测到GPU，DDP脚本将无法运行

✓ torchrun 可用


---
## Cell 0: 全局导入和配置


In [2]:
# ==================== 全局导入 ====================
import os
import re
import json
import time
import random
from pathlib import Path
import subprocess

import numpy as np
import pandas as pd

# Check scipy availability
try:
    from scipy import sparse
    HAS_SCIPY = True
except ImportError:
    HAS_SCIPY = False
    print("警告: scipy 未安装")
    print("  安装命令: pip install scipy")

# PyTorch (用于notebook中的非DDP操作)
import torch

# ==================== 全局配置 ====================
GLOBAL_SEED = 2025
TMP_DIR = Path("./tmp")
TMP_DIR.mkdir(exist_ok=True)
PARQUET_ENGINE = "fastparquet"

# 设置随机种子
def set_seed(seed=2025):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(GLOBAL_SEED)

# I/O辅助函数
def save_parquet_df(df, path):
    df.to_parquet(path, engine=PARQUET_ENGINE, index=False)

def load_parquet_df(path):
    return pd.read_parquet(path, engine=PARQUET_ENGINE)

print("[Setup] ✓ 全局配置完成")
print(f"[Setup] TMP_DIR: {TMP_DIR.absolute()}")
print(f"[Setup] 随机种子: {GLOBAL_SEED}")
if not HAS_SCIPY:
    print("[Setup] ⚠️  scipy 未安装 - Step 3-4及后续步骤需要scipy")

[Setup] ✓ 全局配置完成
[Setup] TMP_DIR: /workspace/recsys/tmp
[Setup] 随机种子: 2025


---
## Step 2: 数据预处理

**执行位置**: Notebook中
**说明**: 加载CSV，清洗文本，解析标签

In [3]:
# ==================== Step 2 参数 ====================
DATA_CSV_PATH = Path("./data/metadata_merged.csv")

TEXT_COLS = ["Title", "Subtitle", "Description", "Slug"]
TAG_COL = "Tags"
ID_COL = "Id"
CREATED_COL_CANDIDATES = ["CreationDate_dt", "CreationDate"]
LAST_ACTIVE_COL_CANDIDATES = ["LastActivityDate_dt", "LastActivityDate"]

TEXT_LOWER = True
TEXT_STRIP_HTML = True
TEXT_DROP_URL = True

DOC_CLEAN_PATH = TMP_DIR / "doc_clean.parquet"
INDEX_MAP_PATH = TMP_DIR / "index_map.parquet"

print("[Step 2] 参数设置完成")

[Step 2] 参数设置完成


In [4]:
# ==================== Step 2 执行 ====================
print(f"[Step 2] 读取CSV: {DATA_CSV_PATH}")

df = pd.read_csv(DATA_CSV_PATH, low_memory=False)
print(f"[Step 2] 加载 {len(df):,} 行")

# 选择时间列
created_col = next((c for c in CREATED_COL_CANDIDATES if c in df.columns), None)
last_active_col = next((c for c in LAST_ACTIVE_COL_CANDIDATES if c in df.columns), None)

# 去重
df = df.drop_duplicates(subset=[ID_COL], keep='first')
print(f"[Step 2] 去重后: {len(df):,} 行")

# 合并文本列
def safe_str(x):
    return "" if pd.isna(x) else str(x)

text_parts = [df[col].apply(safe_str) for col in TEXT_COLS if col in df.columns]
text_all = text_parts[0] if text_parts else pd.Series([""] * len(df))
for part in text_parts[1:]:
    text_all = text_all + " " + part

# 清洗文本
if TEXT_STRIP_HTML:
    text_all = text_all.str.replace(r'<[^>]+>', ' ', regex=True)
if TEXT_DROP_URL:
    text_all = text_all.str.replace(r'https?://\S+', ' ', regex=True)
if TEXT_LOWER:
    text_all = text_all.str.lower()
text_all = text_all.str.replace(r'\s+', ' ', regex=True).str.strip()

# 解析标签
def parse_tags(x):
    if pd.isna(x):
        return ""
    s = str(x).strip('[]"').replace('"', '').replace("'", "")
    tags = [t.strip() for t in s.split(',') if t.strip()]
    return ','.join(tags)

tag_str = df[TAG_COL].apply(parse_tags) if TAG_COL in df.columns else pd.Series([""] * len(df))

# 时间戳
created_at = pd.to_datetime(df[created_col], errors='coerce') if created_col else pd.Series([pd.NaT] * len(df))
last_active_at = pd.to_datetime(df[last_active_col], errors='coerce') if last_active_col else pd.Series([pd.NaT] * len(df))

# 创建doc_df
doc_df = pd.DataFrame({
    'doc_idx': np.arange(len(df), dtype=np.int64),
    'Id': df[ID_COL].values,
    'text_all': text_all.values,
    'tag_str': tag_str.values,
    'created_at': created_at,
    'last_active_at': last_active_at
})

# 统计
print(f"[Step 2] 最终文档数: {len(doc_df):,}")
print(f"[Step 2] 平均文本长度: {doc_df['text_all'].str.len().mean():.1f}")
print(f"[Step 2] 标签覆盖率: {(doc_df['tag_str'].str.len() > 0).sum():,}/{len(doc_df):,}")

# 保存
save_parquet_df(doc_df, DOC_CLEAN_PATH)
index_map = doc_df[['doc_idx', 'Id']]
save_parquet_df(index_map, INDEX_MAP_PATH)

print(f"[Step 2] ✓ 已保存 {DOC_CLEAN_PATH.name}")
print(f"[Step 2] ✓ 已保存 {INDEX_MAP_PATH.name}")

# 预览
display(doc_df.head(3))

[Step 2] 读取CSV: data/metadata_merged.csv
[Step 2] 加载 521,735 行
[Step 2] 去重后: 521,735 行
[Step 2] 最终文档数: 521,735
[Step 2] 平均文本长度: 294.5
[Step 2] 标签覆盖率: 214,603/521,735
[Step 2] ✓ 已保存 doc_clean.parquet
[Step 2] ✓ 已保存 index_map.parquet


,doc_idx,Id,text_all,tag_str,created_at,last_active_at
0,0,6,2013 american community survey find insights i...,"computer science,demographics,social science",2015-07-18 00:51:12,2018-02-05
1,1,7,may 2015 reddit comments get personal with a d...,"internet,linguistics,online communities",2015-08-04 23:59:00,2018-02-06
2,2,8,ocean ship logbooks (1750-1850) explore changi...,"atmospheric science,environment",2015-08-18 21:53:00,2018-01-31


---
## Step 3: Tag视图矩阵构建

**执行位置**: Notebook中
**说明**: 构建Document-Tag稀疏矩阵（Binary, TF-IDF, PPMI）

In [5]:
# ==================== Step 3 参数 ====================
MIN_DF_TAG = 10
MAX_TAGS = None
TFIDF_IDF_SMOOTH = True
ROW_NORM_TFIDF = "l2"
PPMI_EPS = 1e-12
ROW_NORM_PPMI = "l2"

TAG_VOCAB_PATH = TMP_DIR / "tag_vocab.parquet"
DT_BIN_PATH = TMP_DIR / "DT_bin.parquet"
DT_TFIDF_PATH = TMP_DIR / "DT_tfidf.parquet"
DT_PPMI_PATH = TMP_DIR / "DT_ppmi.parquet"

print("[Step 3] 参数设置完成")

[Step 3] 参数设置完成


In [6]:
if not HAS_SCIPY:
    print("❌ 错误: scipy未安装，无法继续")
    print("   安装命令: pip install scipy")
else:
    # ==================== Step 3 执行 ====================
    doc_df = load_parquet_df(DOC_CLEAN_PATH)
    N = len(doc_df)
    print(f"[Step 3] 加载 {N:,} 个文档")

    # 解析标签
    tag_lists = doc_df['tag_str'].str.split(',').apply(lambda x: [t.strip() for t in x if t.strip()])

    # 构建词汇表
    tag_counts = {}
    for tags in tag_lists:
        for tag in tags:
            tag_counts[tag] = tag_counts.get(tag, 0) + 1

    tag_counts = {k: v for k, v in tag_counts.items() if v >= MIN_DF_TAG}
    sorted_tags = sorted(tag_counts.items(), key=lambda x: x[1], reverse=True)
    if MAX_TAGS:
        sorted_tags = sorted_tags[:MAX_TAGS]

    tag_to_idx = {tag: idx for idx, (tag, _) in enumerate(sorted_tags)}
    tag_vocab = pd.DataFrame([
        {'tag_idx': idx, 'tag': tag, 'df': tag_counts[tag]}
        for tag, idx in tag_to_idx.items()
    ]).sort_values('tag_idx')

    save_parquet_df(tag_vocab, TAG_VOCAB_PATH)
    T = len(tag_vocab)
    print(f"[Step 3] 标签词汇表: {T:,} 个标签")

    # 构建D-T二值矩阵
    rows, cols = [], []
    for doc_idx, tags in enumerate(tag_lists):
        for tag in tags:
            if tag in tag_to_idx:
                rows.append(doc_idx)
                cols.append(tag_to_idx[tag])

    vals_bin = np.ones(len(rows), dtype=np.float32)
    DT_bin = sparse.coo_matrix((vals_bin, (rows, cols)), shape=(N, T), dtype=np.float32)
    print(f"[Step 3] D-T 二值矩阵: {DT_bin.nnz:,} 个非零元素")

    # TF-IDF
    print("[Step 3] 计算 TF-IDF...")
    DT_bin_csr = DT_bin.tocsr()  # Convert to CSR for calculations
    tf = DT_bin_csr.copy().astype(np.float32)
    df = np.array(DT_bin_csr.sum(axis=0)).flatten()
    idf = np.log((N + int(TFIDF_IDF_SMOOTH)) / (df + int(TFIDF_IDF_SMOOTH))) + 1.0 if TFIDF_IDF_SMOOTH else np.log(N / np.maximum(df, 1)) + 1.0
    vals_tfidf = tf.data * idf[tf.indices]

    if ROW_NORM_TFIDF == "l2":
        row_norms = np.sqrt(np.array(tf.multiply(tf).sum(axis=1)).flatten())
        row_norms = np.maximum(row_norms, 1e-12)
        vals_tfidf = vals_tfidf / row_norms[DT_bin.row]

    # PPMI
    print("[Step 3] 计算 PPMI...")
    total_cooc = DT_bin_csr.sum()
    row_sums = np.array(DT_bin_csr.sum(axis=1)).flatten()
    col_sums = np.array(DT_bin_csr.sum(axis=0)).flatten()

    p_dt = DT_bin.data / total_cooc
    p_d = row_sums[DT_bin.row] / total_cooc
    p_t = col_sums[DT_bin.col] / total_cooc
    pmi = np.log(p_dt / np.maximum(p_d * p_t, PPMI_EPS))
    vals_ppmi = np.maximum(pmi, 0).astype(np.float32)

    if ROW_NORM_PPMI == "l2":
        DT_ppmi_temp = sparse.coo_matrix((vals_ppmi, (DT_bin.row, DT_bin.col)), shape=(N, T)).tocsr()
        row_norms = np.sqrt(np.array(DT_ppmi_temp.multiply(DT_ppmi_temp).sum(axis=1)).flatten())
        row_norms = np.maximum(row_norms, 1e-12)
        vals_ppmi = vals_ppmi / row_norms[DT_bin.row]

    # 保存
    def save_triplets(rows, cols, vals, path):
        df = pd.DataFrame({'row': rows, 'col': cols, 'val': vals})
        save_parquet_df(df, path)

    save_triplets(DT_bin.row, DT_bin.col, vals_bin, DT_BIN_PATH)
    save_triplets(DT_bin.row, DT_bin.col, vals_tfidf, DT_TFIDF_PATH)
    save_triplets(DT_bin.row, DT_bin.col, vals_ppmi, DT_PPMI_PATH)

    print(f"[Step 3] ✓ 已保存 {DT_BIN_PATH.name}")
    print(f"[Step 3] ✓ 已保存 {DT_TFIDF_PATH.name}")
    print(f"[Step 3] ✓ 已保存 {DT_PPMI_PATH.name}")

[Step 3] 加载 521,735 个文档
[Step 3] 标签词汇表: 394 个标签
[Step 3] D-T 二值矩阵: 445,426 个非零元素
[Step 3] 计算 TF-IDF...
[Step 3] 计算 PPMI...
[Step 3] ✓ 已保存 DT_bin.parquet
[Step 3] ✓ 已保存 DT_tfidf.parquet
[Step 3] ✓ 已保存 DT_ppmi.parquet


---
## Step 4: Text视图矩阵构建

**执行位置**: Notebook中
**说明**: 构建Document-Word矩阵（BM25权重）

In [7]:
# ==================== Step 4 参数 ====================
MIN_DF_W = 200
MAX_DF_RATIO_W = 0.50
KEEP_TOP_W = None
TOKEN_PATTERN = r"[a-z0-9_]+"
MIN_TOKEN_LEN = 2
TO_LOWER = True

STOPWORDS = {
    'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for',
    'of', 'with', 'by', 'from', 'as', 'is', 'was', 'are', 'been', 'be',
    'have', 'has', 'had', 'do', 'does', 'did', 'will', 'would', 'should',
    'could', 'may', 'might', 'can', 'this', 'that', 'these', 'those', 'it'
}

BM25_K1 = 1.5
BM25_B = 0.75
ROW_NORM_BM25 = "l2"

TEXT_VOCAB_PATH = TMP_DIR / "text_vocab.parquet"
DW_BM25_PATH = TMP_DIR / "DW_bm25.parquet"

print("[Step 4] 参数设置完成")

[Step 4] 参数设置完成


In [8]:
if not HAS_SCIPY:
    print("❌ 错误: scipy未安装，无法继续")
    print("   安装命令: pip install scipy")
else:
    # ==================== Step 4 执行 ====================
    def tokenize(s):
        if not isinstance(s, str):
            s = str(s)
        tokens = re.findall(TOKEN_PATTERN, s.lower() if TO_LOWER else s)
        return [t for t in tokens if len(t) >= MIN_TOKEN_LEN and t not in STOPWORDS]

    # 第一遍：构建词汇表
    print("[Step 4] 构建词汇表...")
    word_df = {}
    for idx, row in doc_df.iterrows():
        tokens = set(tokenize(row['text_all']))
        for token in tokens:
            word_df[token] = word_df.get(token, 0) + 1

    max_df = int(N * MAX_DF_RATIO_W)
    word_df = {w: df for w, df in word_df.items() if MIN_DF_W <= df <= max_df}

    sorted_words = sorted(word_df.items(), key=lambda x: x[1], reverse=True)
    if KEEP_TOP_W:
        sorted_words = sorted_words[:KEEP_TOP_W]

    word_to_idx = {word: idx for idx, (word, _) in enumerate(sorted_words)}
    text_vocab = pd.DataFrame([
        {'word_idx': idx, 'word': word, 'df': word_df[word]}
        for word, idx in word_to_idx.items()
    ]).sort_values('word_idx')

    save_parquet_df(text_vocab, TEXT_VOCAB_PATH)
    W = len(text_vocab)
    print(f"[Step 4] 文本词汇表: {W:,} 个单词")

    # 第二遍：构建TF矩阵
    print("[Step 4] 构建D-W TF矩阵...")
    rows, cols, tf_vals = [], [], []
    doc_lens = []

    for idx, row in doc_df.iterrows():
        tokens = tokenize(row['text_all'])
        doc_lens.append(len(tokens))

        tf_counts = {}
        for token in tokens:
            if token in word_to_idx:
                tf_counts[token] = tf_counts.get(token, 0) + 1

        for word, count in tf_counts.items():
            rows.append(idx)
            cols.append(word_to_idx[word])
            tf_vals.append(count)

    tf_vals = np.array(tf_vals, dtype=np.float32)
    doc_lens = np.array(doc_lens, dtype=np.float32)
    print(f"[Step 4] D-W TF: {len(rows):,} 个非零元素")

    # 计算BM25
    print("[Step 4] 计算 BM25...")
    avgdl = doc_lens.mean()
    df_w = text_vocab.set_index('word_idx')['df'].to_dict()
    idf_w = {w_idx: np.log((N - df + 0.5) / (df + 0.5) + 1.0) for w_idx, df in df_w.items()}

    bm25_vals = []
    for r, c, tf in zip(rows, cols, tf_vals):
        dl = doc_lens[r]
        idf = idf_w[c]
        numerator = tf * (BM25_K1 + 1.0)
        denominator = tf + BM25_K1 * (1.0 - BM25_B + BM25_B * dl / avgdl)
        bm25 = idf * numerator / denominator
        bm25_vals.append(bm25)

    bm25_vals = np.array(bm25_vals, dtype=np.float32)

    # 行归一化
    if ROW_NORM_BM25 == "l2":
        DW_temp = sparse.csr_matrix((bm25_vals, (rows, cols)), shape=(N, W))
        row_norms = np.sqrt(np.array(DW_temp.multiply(DW_temp).sum(axis=1)).flatten())
        row_norms = np.maximum(row_norms, 1e-12)
        bm25_vals = bm25_vals / row_norms[np.array(rows)]

    # 保存
    df_out = pd.DataFrame({'row': rows, 'col': cols, 'val': bm25_vals})
    save_parquet_df(df_out, DW_BM25_PATH)

    print(f"[Step 4] ✓ 已保存 {DW_BM25_PATH.name}")

[Step 4] 构建词汇表...
[Step 4] 文本词汇表: 5,739 个单词
[Step 4] 构建D-W TF矩阵...
[Step 4] D-W TF: 7,983,559 个非零元素
[Step 4] 计算 BM25...
[Step 4] ✓ 已保存 DW_bm25.parquet


---
## Step 5: 随机游走参数准备

**执行位置**: Notebook中
**说明**: 保存随机游走参数（实际游走在Step 6的DDP脚本中进行）

In [9]:
# ==================== Step 5 参数 ====================
RW_WALKS_PER_DOC = 10
RW_L_DOCS_PER_SENT = 40
RW_SEED_BASE = 2025
RW_AVOID_BACKTRACK = True
RW_RESTART_PROB = 0.15
RW_X_DEGREE_POW = -0.5
RW_X_NO_REPEAT_LAST = 1

RW_PARAMS_PATH = TMP_DIR / "rw_params.parquet"

# 保存参数
rw_params = pd.DataFrame([{
    'RW_WALKS_PER_DOC': RW_WALKS_PER_DOC,
    'RW_L_DOCS_PER_SENT': RW_L_DOCS_PER_SENT,
    'RW_SEED_BASE': RW_SEED_BASE,
    'RW_AVOID_BACKTRACK': RW_AVOID_BACKTRACK,
    'RW_RESTART_PROB': RW_RESTART_PROB,
    'RW_X_DEGREE_POW': RW_X_DEGREE_POW,
    'RW_X_NO_REPEAT_LAST': RW_X_NO_REPEAT_LAST
}])

save_parquet_df(rw_params, RW_PARAMS_PATH)
print(f"[Step 5] ✓ 已保存随机游走参数到 {RW_PARAMS_PATH.name}")

[Step 5] ✓ 已保存随机游走参数到 rw_params.parquet


---
## Step 6: SGNS训练 🚀 **DDP脚本调用**

**执行位置**: DDP脚本 (`ddp_scripts/step6_sgns_training_ddp.py`)
**说明**: 使用torchrun启动多GPU训练

**重要**: 这是最耗时的步骤，建议使用所有可用GPU

In [10]:
# ==================== Step 6: 调用DDP脚本 ====================
import subprocess
import sys

# 检测GPU数量
NUM_GPUS = torch.cuda.device_count()
if NUM_GPUS == 0:
    print("❌ 错误: 未检测到GPU，无法运行DDP训练")
    print("   请在有GPU的环境中运行此步骤")
else:
    print(f"[Step 6] 检测到 {NUM_GPUS} 个GPU")
    print(f"[Step 6] 准备启动SGNS训练...")

    # 构建torchrun命令
    cmd = [
        "torchrun",
        f"--nproc_per_node={NUM_GPUS}",
        "ddp_scripts/step6_sgns_training_ddp.py",
        "--tmp_dir", "./tmp",
        "--epochs", "4",
        "--dim", "256",
        "--neg", "10",
        "--lr", "0.025",
        "--amp", "true",
        "--tf32", "true",
        "--optimizer", "sgd",
        "--sparse", "false",
        "--window_tag", "5",
        "--keep_prob_tag", "1.0",
        "--forward_only_tag", "false",
        "--ctx_cap_tag", "0",
        "--batch_pairs_tag", "204800",
        "--max_pairs_tag", "20000000",
        "--window_text", "4",
        "--keep_prob_text", "0.35",
        "--forward_only_text", "true",
        "--ctx_cap_text", "4",
        "--batch_pairs_text", "204800",
        "--max_pairs_text", "20000000",
        "--accum", "1",
        "--log_every", "200",
        "--save_epoch_emb", "true",
    ]

    print(f"\n[Step 6] 执行命令:")
    print(f"  {' '.join(cmd)}\n")
    print("=" * 80)

    # 执行命令
    try:
        result = subprocess.run(cmd, check=True)
        print("=" * 80)
        print(f"\n[Step 6] ✓ SGNS训练完成!")
        print(f"[Step 6] ✓ 输出文件: tmp/Z_tag.parquet, tmp/Z_text.parquet")
    except subprocess.CalledProcessError as e:
        print("=" * 80)
        print(f"\n[Step 6] ❌ 训练失败: {e}")
        print("请检查错误信息并重试")
    except FileNotFoundError:
        print("=" * 80)
        print(f"\n[Step 6] ❌ 找不到torchrun命令")
        print("请确保PyTorch已正确安装: pip install torch --upgrade")

# 验证输出
Z_TAG_PATH = TMP_DIR / "Z_tag.parquet"
Z_TEXT_PATH = TMP_DIR / "Z_text.parquet"

if Z_TAG_PATH.exists() and Z_TEXT_PATH.exists():
    z_tag = load_parquet_df(Z_TAG_PATH)
    z_text = load_parquet_df(Z_TEXT_PATH)
    print(f"\n[验证] Z_tag shape: {z_tag.shape}")
    print(f"[验证] Z_text shape: {z_text.shape}")
    display(z_tag.head(3))
else:
    print(f"\n[警告] 输出文件不存在，请检查训练是否成功")

❌ 错误: 未检测到GPU，无法运行DDP训练
   请在有GPU的环境中运行此步骤


/opt/conda/envs/recsys-gpu/lib/python3.10/site-packages/torch/cuda/__init__.py:1007: UserWarning: Can't initialize NVML
  raw_cnt = _raw_device_count_nvml()



[验证] Z_tag shape: (521735, 257)
[验证] Z_text shape: (521735, 257)


,doc_idx,f0,f1,f2,f3,f4,f5,f6,f7,f8,...,f246,f247,f248,f249,f250,f251,f252,f253,f254,f255
0,0,0.056348,-0.070534,-0.046454,0.026976,0.045981,-0.019534,0.072503,0.101383,-0.000926,...,0.060767,0.070982,-0.048615,-0.030430,0.087409,-0.059957,-0.040102,0.108852,-0.043066,0.023502
1,1,0.043500,0.099737,-0.095901,0.095466,-0.007472,0.090022,-0.051307,0.073374,-0.030511,...,0.099195,-0.030271,0.059702,0.039583,0.071028,-0.072855,-0.020589,0.050518,-0.076593,0.092885
2,2,0.043283,-0.017254,0.038883,-0.074650,0.042688,0.078111,0.103584,-0.110195,0.008677,...,0.054053,-0.031277,0.076465,-0.062687,0.020501,-0.045325,0.006400,0.101797,0.021225,0.020413


---
## Step 7: FAISS ANN图构建 🚀 **DDP脚本调用**

**执行位置**: DDP脚本 (`ddp_scripts/step7_faiss_ann_ddp.py`)
**说明**: 使用FAISS构建k-NN相似度图

In [12]:
# ==================== Step 7: 调用DDP脚本 ====================
if NUM_GPUS == 0:
    print("❌ 错误: 未检测到GPU")
else:
    print(f"[Step 7] 准备启动FAISS ANN构建...")

    cmd = [
        "torchrun",
        f"--nproc_per_node={NUM_GPUS}",
        "ddp_scripts/step7_faiss_ann_ddp.py",
        "--tmp_dir", "./tmp",
        "--k", "50",
        "--batch_q", "8192",
        "--part_size", "2000000",
        "--use_gpu", "true",
        "--index_type", "flat_ip",
    ]

    print(f"\n[Step 7] 执行命令:")
    print(f"  {' '.join(cmd)}\n")
    print("=" * 80)

    try:
        result = subprocess.run(cmd, check=True)
        print("=" * 80)
        print(f"\n[Step 7] ✓ FAISS ANN构建完成!")
    except subprocess.CalledProcessError as e:
        print("=" * 80)
        print(f"\n[Step 7] ❌ 构建失败: {e}")
    except FileNotFoundError:
        print("=" * 80)
        print(f"\n[Step 7] ❌ 找不到torchrun命令")

# 验证输出
tag_manifest = TMP_DIR / "S_tag_topk_k50_manifest.json"
text_manifest = TMP_DIR / "S_text_topk_k50_manifest.json"

if tag_manifest.exists() and text_manifest.exists():
    with open(tag_manifest) as f:
        tag_info = json.load(f)
    with open(text_manifest) as f:
        text_info = json.load(f)

    print(f"\n[验证] Tag图: {tag_info['total_edges']:,} 条边, {tag_info['num_parts']} 个分区")
    print(f"[验证] Text图: {text_info['total_edges']:,} 条边, {text_info['num_parts']} 个分区")
else:
    print(f"\n[警告] Manifest文件不存在")

❌ 错误: 未检测到GPU


KeyError: 'num_parts'

---
## Step 8-9: 图对称化和融合

**执行位置**: Notebook中
**说明**: 对称化、归一化、融合Tag和Text视图

In [ ]:
# ==================== Step 8-9 参数 ====================
SYM_METHOD = "max"
ROW_NORM_EPS = 1e-12
K_FUSED = 50
W_TAG_GLOBAL = 1.0
W_TEXT_GLOBAL = 1.0
SAVE_PART_EDGES = 2_000_000

print("[Step 8-9] 参数设置完成")

In [ ]:
if not HAS_SCIPY:
    print("❌ 错误: scipy未安装，无法继续")
    print("   安装命令: pip install scipy")
else:
    # ==================== Step 8-9 执行 ====================

    # 辅助函数
    def load_manifest(prefix, k):
        manifest_path = TMP_DIR / f"{prefix}_k{k}_manifest.json"
        with open(manifest_path, 'r') as f:
            return json.load(f)

    def load_partitioned_edges(prefix, k):
        manifest = load_manifest(prefix, k)
        dfs = []
        for part_file in manifest['part_files']:
            df = load_parquet_df(TMP_DIR / part_file)
            dfs.append(df)
        return pd.concat(dfs, ignore_index=True)

    def sym_and_rownorm(rows, cols, vals, N, method, eps):
        mat = sparse.coo_matrix((vals, (rows, cols)), shape=(N, N)).tocsr()
        if method == "max":
            mat = mat.maximum(mat.T)
        elif method == "avg":
            mat = (mat + mat.T) / 2.0
        mat = mat.tocsr()

        row_sums = np.array(mat.sum(axis=1)).flatten()
        row_sums = np.maximum(row_sums, eps)
        row_scale = 1.0 / row_sums

        mat_norm = mat.copy()
        for i in range(N):
            start, end = mat_norm.indptr[i], mat_norm.indptr[i + 1]
            mat_norm.data[start:end] *= row_scale[i]

        mat_coo = mat_norm.tocoo()
        return mat_coo.row, mat_coo.col, mat_coo.data

    def save_partitioned_edges(rows, cols, vals, N, prefix, k, part_size):
        df = pd.DataFrame({'row': rows, 'col': cols, 'val': vals})
        num_parts = (len(df) + part_size - 1) // part_size
        manifest = {
            'N': int(N),
            'k': int(k),
            'total_edges': len(df),
            'num_parts': num_parts,
            'part_files': []
        }

        for i in range(num_parts):
            start = i * part_size
            end = min((i + 1) * part_size, len(df))
            part_df = df.iloc[start:end]
            part_file = f"{prefix}_k{k}_part{i:04d}.parquet"
            part_path = TMP_DIR / part_file
            save_parquet_df(part_df, part_path)
            manifest['part_files'].append(part_file)

        manifest_path = TMP_DIR / f"{prefix}_k{k}_manifest.json"
        with open(manifest_path, 'w') as f:
            json.dump(manifest, f, indent=2)
        return manifest_path

    # Step 8: 对称化
    print("[Step 8] 对称化Tag图...")
    df_tag = load_partitioned_edges("S_tag_topk", 50)
    rows, cols, vals = sym_and_rownorm(
        df_tag['row'].values, df_tag['col'].values, df_tag['val'].values,
        N, SYM_METHOD, ROW_NORM_EPS
    )
    save_partitioned_edges(rows, cols, vals, N, "S_tag_symrow", 50, SAVE_PART_EDGES)
    print(f"[Step 8] Tag对称化: {len(rows):,} 条边")

    print("[Step 8] 对称化Text图...")
    df_text = load_partitioned_edges("S_text_topk", 50)
    rows, cols, vals = sym_and_rownorm(
        df_text['row'].values, df_text['col'].values, df_text['val'].values,
        N, SYM_METHOD, ROW_NORM_EPS
    )
    save_partitioned_edges(rows, cols, vals, N, "S_text_symrow", 50, SAVE_PART_EDGES)
    print(f"[Step 8] Text对称化: {len(rows):,} 条边")

    # Step 9: 融合
    print("[Step 9] 融合Tag + Text视图...")

    def fuse_two_views(df1, df2, N, w1, w2, eps):
        mat1 = sparse.coo_matrix((df1['val'], (df1['row'], df1['col'])), shape=(N, N)).tocsr()
        mat2 = sparse.coo_matrix((df2['val'], (df2['row'], df2['col'])), shape=(N, N)).tocsr()

        row_norms1 = np.array(mat1.multiply(mat1).sum(axis=1)).flatten()
        row_norms2 = np.array(mat2.multiply(mat2).sum(axis=1)).flatten()

        alpha = row_norms1 / np.maximum(row_norms1 + row_norms2, eps)
        beta = row_norms2 / np.maximum(row_norms1 + row_norms2, eps)
        alpha *= w1
        beta *= w2

        mat_fused = sparse.csr_matrix((N, N), dtype=np.float32)
        for i in range(N):
            row1 = mat1[i].toarray().flatten()
            row2 = mat2[i].toarray().flatten()
            row_fused = alpha[i] * row1 + beta[i] * row2
            mat_fused[i] = row_fused

        mat_coo = mat_fused.tocoo()
        return mat_coo.row, mat_coo.col, mat_coo.data

    df_tag_sym = load_partitioned_edges("S_tag_symrow", 50)
    df_text_sym = load_partitioned_edges("S_text_symrow", 50)

    rows, cols, vals = fuse_two_views(
        df_tag_sym, df_text_sym, N, W_TAG_GLOBAL, W_TEXT_GLOBAL, ROW_NORM_EPS
    )

    # 保留top-K
    df_fused = pd.DataFrame({'row': rows, 'col': cols, 'val': vals})
    df_fused = df_fused.sort_values(['row', 'val'], ascending=[True, False])
    df_fused = df_fused.groupby('row').head(K_FUSED)

    save_partitioned_edges(
        df_fused['row'].values, df_fused['col'].values, df_fused['val'].values,
        N, "S_fused_symrow", K_FUSED, SAVE_PART_EDGES
    )

    print(f"[Step 9] ✓ 融合完成: {len(df_fused):,} 条边")
    print(f"[Step 9] ✓ 最终图已保存")

---
## 验证和可视化

查看最终融合图的统计信息

In [ ]:
# 加载最终融合图
final_manifest = TMP_DIR / "S_fused_symrow_k50_manifest.json"

if final_manifest.exists():
    with open(final_manifest) as f:
        info = json.load(f)

    print("=" * 60)
    print("最终融合图统计")
    print("=" * 60)
    print(f"节点数: {info['N']:,}")
    print(f"边数: {info['total_edges']:,}")
    print(f"平均度数: {info['total_edges'] / info['N']:.2f}")
    print(f"分区数: {info['num_parts']}")
    print(f"K值: {info['k']}")
    print("=" * 60)

    # 加载一个分区查看
    df_sample = load_partitioned_edges("S_fused_symrow", 50)
    print(f"\n样本数据 (前10行):")
    display(df_sample.head(10))

    # 统计
    degree_dist = df_sample.groupby('row').size()
    print(f"\n度数分布:")
    print(f"  最小度数: {degree_dist.min()}")
    print(f"  最大度数: {degree_dist.max()}")
    print(f"  平均度数: {degree_dist.mean():.2f}")
    print(f"  中位数度数: {degree_dist.median():.1f}")
else:
    print("❌ 最终图未找到，请确保前面步骤都已成功执行")

---
## 📋 Pipeline总结

### 已完成的步骤

- [x] **Step 2**: 数据预处理 (Notebook)
- [x] **Step 3**: Tag视图矩阵 (Notebook)
- [x] **Step 4**: Text视图矩阵 (Notebook)
- [x] **Step 5**: 随机游走参数 (Notebook)
- [x] **Step 6**: SGNS训练 (**DDP脚本** 🚀)
- [x] **Step 7**: FAISS ANN (** DDP脚本** 🚀)
- [x] **Step 8-9**: 图处理和融合 (Notebook)

### 输出文件

| 文件 | 描述 |
|------|------|
| `tmp/doc_clean.parquet` | 清洗后的文档 |
| `tmp/tag_vocab.parquet` | 标签词汇表 |
| `tmp/text_vocab.parquet` | 文本词汇表 |
| `tmp/DT_ppmi.parquet` | Doc-Tag PPMI矩阵 |
| `tmp/DW_bm25.parquet` | Doc-Word BM25矩阵 |
| `tmp/Z_tag.parquet` | Tag嵌入 (256维) |
| `tmp/Z_text.parquet` | Text嵌入 (256维) |
| `tmp/S_fused_symrow_k50_*.parquet` | 最终融合图 |

### DDP脚本使用说明

#### 独立运行Step 6训练:
```bash
torchrun --nproc_per_node=4 ddp_scripts/step6_sgns_training_ddp.py \
  --tmp_dir ./tmp --epochs 4 --dim 256 --neg 10
```

#### 独立运行Step 7 ANN:
```bash
torchrun --nproc_per_node=4 ddp_scripts/step7_faiss_ann_ddp.py \
  --tmp_dir ./tmp --k 50
```

### 架构优势

✅ **Notebook交互性**: 可以随时查看中间结果、调整参数
✅ **DDP高性能**: GPU密集型步骤使用完整的多GPU并行
✅ **灵活调试**: 可以单独运行任意步骤
✅ **生产就绪**: DDP脚本可直接用于服务器批量训练

---

**🎉 Pipeline完成！**
